# AnyTime Universal Benchmark

Run early-exit benchmark for any AnyTime model. Switch by changing **one import line** below.

All HW metrics **per-PID** (process-scoped). Energy = `Î£ (power_w_i Ã— end_to_end_sec_i)` in Joules.

Per-run dir contains:
- `hw_results.json` â€” latency + memory + energy + per-PID GPU/CPU + FLOPs/params
- `quality_results.json` â€” accuracy / F1 / mAP / PPL (only if `skip_quality=False`)

Default sweep = HW-only. Pass `skip_quality=False` to enable quality eval.

Outputs:
- `logs/benchmark/{model}/...` â€” per-run JSONs (nested 2-3 levels deep)
- `results/{model}/` â€” CSV summaries


## 1. Setup

In [1]:
import numpy as np

In [2]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

repo root: D:\Research\earlyexit\EarlyExitMonoRepo


In [3]:
# HF login (only needed if pulling from a private repo)
from shared import load_env

load_env()

try:
    from google.colab import userdata

    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN", ""))
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF: logged in")
else:
    print("HF: no token, public repos only")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF: logged in


## 2. Choose model

**Change ONE import line below** to swap which model gets benchmarked.

In [4]:
# # === SWAP THIS LINE TO PICK MODEL ===
# # from benchmark_config import bert as cfg
# from benchmark_config import llama  as cfg
# # from benchmark_config import vision as cfg
# # from benchmark_config import yolo   as cfg

# print("model :", cfg.NAME)
# print("family:", cfg.MODEL_FAMILY)
# print("out   :", cfg.OUT_DIR)

## 3. Run full sweep

Default = HW-only (`skip_quality=True`). Each run does:
- `profile_hw()` â€” HW + latency + per-PID power/VRAM/CPU + FLOPs/params + EDP
- `evaluate_quality()` â€” (skipped by default)

Override sweep dims via `only_*` kwargs (see `cfg.run_all` signature).


In [5]:
# cfg.run_all(skip_quality=False)          # HW + quality (all datasets)
# # cfg.run_all()                            # HW-only (cnn_dailymail latency only)
# # cfg.run_all(skip_quality=False, only_exit=4)  # single exit smoke test

## 3b. Run ALL models (full sweep)

In [ ]:
# # Run ALL models -- HW + quality sweep for every model family.
# # Comment out any model you want to skip.
# from benchmark_config import bert, llama, vision, yolo

# for _cfg in [bert, llama, vision, yolo]:
#     print(f"{60*chr(61)}")
#     print(f"  {_cfg.NAME} ({_cfg.MODEL_FAMILY})")
#     print(60*chr(61))
#     try:
#         _cfg.run_all(skip_quality=False,dry_run=True)  # HW + quality (all datasets)
#     except Exception as _e:
#         print(f"[{_cfg.NAME}] run_all failed: {_e}")
#         import traceback; traceback.print_exc()


In [ ]:
from shared import write_benchmark_csvs, average_across_tasks
from shared.averager import write_averaged_json
from benchmark_config import bert, llama, vision, yolo
from collections import defaultdict
import json
import pandas as pd

for _cfg in [bert, llama, vision, yolo]:
    out_dir = Path(_cfg.OUT_DIR)
    csv_root = REPO_ROOT / "results" / _cfg.NAME
    csv_root.mkdir(parents=True, exist_ok=True)

    hw_dirs = {p.parent for p in out_dir.rglob("hw_results.json")}
    q_dirs = {p.parent for p in out_dir.rglob("quality_results.json")}
    run_dirs = list(hw_dirs | q_dirs)
    if not run_dirs:
        print(f"[{_cfg.NAME}] no results found, skipping")
        continue
    print(f"[{_cfg.NAME}] found {len(run_dirs)} run dirs ({len(hw_dirs)} hw, {len(q_dirs)} quality)")

    # group by dataset parent dir; each group becomes one CSV bundle
    groups = defaultdict(dict)
    dataset_dirs = defaultdict(set)         # dataset_name -> {parent_dir_of_exits}
    for rd in run_dirs:
        rel_parent = rd.parent.relative_to(out_dir).as_posix()
        group_key = rel_parent.replace("/", "_") or "root"
        groups[group_key][rd.name] = rd
        dataset_dirs[group_key].add(rd.parent)

    # 1. per-dataset CSVs (one bundle per dataset)
    for group_key, runs in sorted(groups.items()):
        csv_dir = csv_root / group_key
        csv_dir.mkdir(parents=True, exist_ok=True)
        write_benchmark_csvs(
            results_files=runs,
            out_dir=csv_dir,
            baseline_key=None,
            method_order=sorted(runs.keys()),
        )
        print(f"  {group_key}: {len(runs)} runs -> {csv_dir}")

    # 2. cross-dataset averaged CSV (mean per method across all datasets)
    if len(groups) > 1:
        per_task = {k: next(iter(dirs)) for k, dirs in dataset_dirs.items()}
        averaged = average_across_tasks(per_task, run_pattern="*")
        if averaged:
            avg_root = csv_root / "_averaged"
            avg_root.mkdir(parents=True, exist_ok=True)
            # synthesize per-method hw_results.json so write_benchmark_csvs accepts it
            synth_runs = {}
            for method, metrics in averaged.items():
                md = avg_root / method
                md.mkdir(parents=True, exist_ok=True)
                (md / "hw_results.json").write_text(
                    json.dumps({"aggregate": metrics}, indent=2)
                )
                synth_runs[method] = md
            write_benchmark_csvs(
                results_files=synth_runs,
                out_dir=avg_root,
                baseline_key=None,
                method_order=sorted(synth_runs.keys()),
            )
            print(f"  _averaged: {len(averaged)} methods across {len(groups)} datasets -> {avg_root}")

print("All CSVs under:", REPO_ROOT / "results")


## 4b. Export CSVs — ALL models

## 4. Export CSVs

Recursive walk over per-run dirs (handles 2-3 level nesting per model). Merges
`hw_results.json` + `quality_results.json` if both exist. Groups by parent dir.


In [ ]:
from shared import write_benchmark_csvs
from collections import defaultdict

out_dir = Path(cfg.OUT_DIR)
csv_root = REPO_ROOT / "results" / cfg.NAME
csv_root.mkdir(parents=True, exist_ok=True)

hw_dirs = {p.parent for p in out_dir.rglob("hw_results.json")}
q_dirs = {p.parent for p in out_dir.rglob("quality_results.json")}
run_dirs = list(hw_dirs | q_dirs)
print(f"found {len(run_dirs)} run dirs under {out_dir} ({len(hw_dirs)} hw, {len(q_dirs)} quality)")

groups = defaultdict(dict)
for rd in run_dirs:
    rel_parent = rd.parent.relative_to(out_dir).as_posix()
    group_key = rel_parent.replace("/", "_") or "root"
    groups[group_key][rd.name] = rd

for group_key, runs in sorted(groups.items()):
    csv_dir = csv_root / group_key
    csv_dir.mkdir(parents=True, exist_ok=True)
    write_benchmark_csvs(
        results_files=runs,
        out_dir=csv_dir,
        baseline_key=None,
        method_order=sorted(runs.keys()),
    )
    print(f"  wrote {len(runs)} runs -> {csv_dir}")

print("CSVs under:", csv_root)


## 5. Quick view

In [9]:
import pandas as pd

for csv_name in ['latency.csv', 'energy.csv', 'quality.csv', 'hardware.csv']:
    for f in csv_root.rglob(csv_name):
        rel = f.relative_to(csv_root)
        print(f'=== {rel} ===')
        print(pd.read_csv(f).head(8))
        print()


In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd

# Plot every backend × every dataset group, plus the cross-dataset _averaged/.
# Writes PNGs next to each CSV bundle: results/{backend}/{group}/plots/*.png

def _exit_key(method: str):
    nums = [int(n) for n in re.findall(r"\d+", method)]
    return nums or [10**9]

def _plot_metric(df, x_col, y_cols, title, ylabel, out_path):
    fig, ax = plt.subplots(figsize=(7, 4))
    plotted = 0
    for y in y_cols:
        if y not in df.columns: continue
        ax.plot(df[x_col], df[y], marker="o", label=y); plotted += 1
    ax.set_xlabel("exit"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if plotted > 1: ax.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    fig.savefig(out_path, dpi=120); plt.show(); plt.close(fig)

def _pareto(lat, qua, out_path, title):
    if lat is None or qua is None: return
    merged = lat.merge(qua, on="method", how="inner")
    if merged.empty: return
    y = next((c for c in ("map","map50","acc","top1_acc","f1","glue_score") if c in merged.columns), None)
    if y is None: return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(merged["end_to_end_sec_mean"]*1000, merged[y], s=40)
    for _, row in merged.iterrows():
        ax.annotate(row["method"], (row["end_to_end_sec_mean"]*1000, row[y]),
                    fontsize=7, xytext=(3,3), textcoords="offset points")
    ax.set_xlabel("end-to-end latency (ms)"); ax.set_ylabel(y); ax.set_title(title)
    ax.grid(True, alpha=0.3); plt.tight_layout()
    fig.savefig(out_path, dpi=120); plt.show(); plt.close(fig)

results_root = REPO_ROOT / "results"
for model_dir in sorted(p for p in results_root.iterdir() if p.is_dir()):
    for group_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
        plots_dir = group_dir / "plots"
        plots_dir.mkdir(parents=True, exist_ok=True)
        tag = f"{model_dir.name}/{group_dir.name}"

        def _load(name):
            p = group_dir / name
            if not p.exists(): return None
            df = pd.read_csv(p)
            if "method" in df.columns:
                df = df.sort_values("method", key=lambda s: s.map(_exit_key)).reset_index(drop=True)
            return df

        lat = _load("latency.csv")
        eng = _load("energy.csv")
        qua = _load("quality.csv")
        hwd = _load("hardware.csv")

        if lat is not None:
            print(f"=== {tag} :: latency ===")
            _plot_metric(lat, "method",
                ["end_to_end_sec_mean","ttft_sec_mean"]   # e2e is universal (== forward for non-LLM); ttft only LLM,
                f"{tag} latency", "seconds", plots_dir / "latency.png")
        if eng is not None:
            print(f"=== {tag} :: energy ===")
            _plot_metric(eng, "method",
                ["joules_per_sample","avg_power_w"],
                f"{tag} energy", "J/sample (and W)", plots_dir / "energy.png")
        if qua is not None:
            q_cols = [c for c in qua.columns if c != "method"]
            print(f"=== {tag} :: quality ===")
            _plot_metric(qua, "method", q_cols,
                f"{tag} quality", "metric", plots_dir / "quality.png")
        if hwd is not None:
            print(f"=== {tag} :: hardware ===")
            _plot_metric(hwd, "method",
                ["avg_vram_allocated_mb","avg_ram_used_mb","avg_proc_gpu_util_pct","avg_cpu_cores_used"],
                f"{tag} hardware", "value", plots_dir / "hardware.png")

        _pareto(lat, qua, plots_dir / "pareto.png", f"{tag} quality vs latency")

print("plots written under", results_root)
